# The confidence of REGAIN predictions for spatial transcriptomes was obtained using the Ridge Error Regression model.



In [ ]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib
import os
rootpath = os.getcwd()
workpath = os.path.join(rootpath, "..")
os.chdir(workpath)

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

# ======================
# Paths
# ======================
train_emb_path = Path('experiments/demo/val_output/epoch_1_embedding.csv')
train_err_path = Path('experiments/demo/val_output/epoch_1_error.csv')
infer_emb_path = Path('experiments/demo/predict_output/epoch_demo_predict_embedding.csv')
confidence_out_path = Path('experiments/demo/predict_output/epoch_demo_predict_confidence.csv')

model_dir = Path('ridge_mode/demo')
model_path = model_dir / 'ridge_model.pkl'
scaler_path = model_dir / 'scaler.pkl'
metadata_path = model_dir / 'ridge_train_metadata.json'

model_dir.mkdir(parents=True, exist_ok=True)
confidence_out_path.parent.mkdir(parents=True, exist_ok=True)

alpha = 1.0
max_iter = 1000
tol = 1e-4
random_state = 42
train_fraction = 0.20

In [ ]:
def normalize_id_column(df, id_col='cell_id'):
    df = df.copy()
    first_col = df.columns[0]
    if id_col in df.columns:
        pass
    elif first_col.startswith('Unnamed') or first_col in {'spot_id', 'barcode', 'id'}:
        df = df.rename(columns={first_col: id_col})
    else:
        df = df.rename(columns={first_col: id_col})
    return df


def numeric_feature_columns(df, id_col='cell_id'):
    cols = []
    for col in df.columns:
        if col == id_col:
            continue
        if pd.api.types.is_numeric_dtype(df[col]):
            cols.append(col)
    return cols


def load_embedding(path, id_col='cell_id'):
    df = normalize_id_column(pd.read_csv(path), id_col=id_col)
    df[id_col] = df[id_col].astype(str)
    emb_cols = numeric_feature_columns(df, id_col=id_col)
    if len(emb_cols) == 0:
        raise ValueError(f'No numeric embedding columns found in {path}')
    return df[[id_col] + emb_cols].copy(), emb_cols


def load_error(path, id_col='cell_id'):
    df = normalize_id_column(pd.read_csv(path), id_col=id_col)
    df[id_col] = df[id_col].astype(str)
    err_cols = numeric_feature_columns(df, id_col=id_col)
    if len(err_cols) == 0:
        raise ValueError(f'No numeric error columns found in {path}')
    return df[[id_col] + err_cols].copy(), err_cols


In [ ]:
df_emb, emb_cols = load_embedding(train_emb_path)
df_err, err_cols = load_error(train_err_path)

df_train = df_emb.merge(df_err, on='cell_id', how='inner', validate='one_to_one')
if df_train.empty:
    raise ValueError('No matched rows between embedding and error by cell_id')

X = df_train[emb_cols].to_numpy(dtype=np.float32)
Y = df_train[err_cols].to_numpy(dtype=np.float32)

y = np.nanmean(Y, axis=1)
valid = np.isfinite(X).all(axis=1) & np.isfinite(y)

X_all_valid = X[valid]
y_all_valid = y[valid]
ids_all_valid = df_train.loc[valid, 'cell_id'].to_numpy()

rng = np.random.default_rng(random_state)
n_train = max(1, int(np.ceil(len(ids_all_valid) * train_fraction)))
train_idx = np.sort(rng.choice(len(ids_all_valid), size=n_train, replace=False))

X = X_all_valid[train_idx]
y = y_all_valid[train_idx]
train_ids = ids_all_valid[train_idx]



In [ ]:
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

model = Ridge(
    alpha=alpha,
    fit_intercept=True,
    max_iter=max_iter,
    tol=tol,
    random_state=random_state,
)
model.fit(X_scaled, y)

train_pred = model.predict(X_scaled)
mae = mean_absolute_error(y, train_pred)
rmse = float(np.sqrt(mean_squared_error(y, train_pred)))
r2 = r2_score(y, train_pred)


In [ ]:
joblib.dump(model, model_path)
joblib.dump(scaler, scaler_path)

metadata = {
    'train_emb_path': str(train_emb_path),
    'train_err_path': str(train_err_path),
    'infer_emb_path': str(infer_emb_path),
    'confidence_out_path': str(confidence_out_path),
    'model_path': str(model_path),
    'scaler_path': str(scaler_path),
    'n_train_rows': int(len(train_ids)),
    'train_fraction': float(train_fraction),
    'n_valid_aligned_rows': int(len(ids_all_valid)),
    'embedding_dim': int(X.shape[1]),
    'n_error_genes': int(len(err_cols)),
    'alpha': float(alpha),
    'max_iter': int(max_iter),
    'tol': float(tol),
    'random_state': int(random_state),
    'train_mae': float(mae),
    'train_rmse': float(rmse),
    'train_r2': float(r2),
    'embedding_columns': [str(c) for c in emb_cols],
    'error_columns': [str(c) for c in err_cols],
}
metadata_path.write_text(json.dumps(metadata, indent=2, ensure_ascii=False), encoding='utf-8')


In [ ]:
test_emb, test_emb_cols = load_embedding(infer_emb_path)

missing_cols = [c for c in emb_cols if c not in test_emb.columns]
if missing_cols:
    raise ValueError(f'Inference embedding is missing training columns: {missing_cols[:10]}')

cell_ids = test_emb['cell_id'].to_numpy()
X_test = test_emb[emb_cols].to_numpy(dtype=np.float32)
valid_test = np.isfinite(X_test).all(axis=1)

loaded_model = joblib.load(model_path)
loaded_scaler = joblib.load(scaler_path)

error_pred = np.full(X_test.shape[0], np.nan, dtype=np.float32)
error_pred[valid_test] = loaded_model.predict(loaded_scaler.transform(X_test[valid_test])).astype(np.float32)

uncertainty_value = float(np.nanstd(error_pred))
uncertainty = np.full_like(error_pred, uncertainty_value, dtype=np.float32)
confidence = np.exp(-error_pred)
confidence_with_unc = np.exp(-(error_pred + uncertainty))


In [ ]:
df_out = pd.DataFrame({
    'cell_id': cell_ids,
    'pred_error': error_pred,
    'uncertainty': uncertainty,
    'confidence': confidence,
    'confidence_with_unc': confidence_with_unc,
})

df_out.to_csv(confidence_out_path, index=False)
print('Saved confidence:', confidence_out_path)
df_out.head()
